# Open Router 

Le vamos a hablar a un model por API.
- Documentación: https://openrouter.ai/docs/quickstart
- Lista de modelos y precios: https://openrouter.ai/models

**Necesitas una API key**. Con un modelo `:free` no necesitas tarjeta.

In [ ]:
import os
import json
import time

import httpx

BASE_URL = "https://openrouter.ai/api/v1"
MODELO_FREE = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"

# La API key es un SECRETO: no la escribas aquí ni la subas a git.
# Copia .env.example a .env y pon tu key ahí.
try:
    from dotenv import load_dotenv

    load_dotenv()
except ModuleNotFoundError:
    raise

API_KEY = os.environ.get("OPENROUTER_API_KEY")
assert API_KEY, "Falta OPENROUTER_API_KEY"
print(API_KEY)

## Sostener el hilo: la conversación a mano

La API es **stateless**: no recuerda el mensaje anterior. Para sostener una
conversación reenvías **toda la lista `messages`** en cada llamada.

Primero encapsulamos la llamada en una función `preguntar(messages)` que
devuelve solo el texto de la respuesta.

In [ ]:
def preguntar(messages: list[dict], modelo: str = MODELO_FREE) -> str:
    """Manda la lista completa de mensajes y devuelve el texto de la respuesta."""
    r = httpx.post(
        f"{BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={"model": modelo, "messages": messages},
        timeout=60,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]

In [ ]:
# 1ª llamada
messages = [                                
    {"role": "system", "content": "Eres un tutor breve. Responde en 1-2 frases."},
    {"role": "user", "content": "¿Qué es una API?"},
]
respuesta = preguntar(messages)          
print("bot>", respuesta)
messages.append({"role": "assistant", "content": respuesta}) # Guardamos la respuesta del asistente

# 2ª llamada: se manda TODO otra vez
messages.append({"role": "user", "content": "Dame un ejemplo real."})
respuesta = preguntar(messages)           
print("bot>", respuesta)

In [ ]:
# La lista completa es toda la "memoria" que hay: crece con cada turno
print(json.dumps(messages, indent=2, ensure_ascii=False))